# 03 · Unsupervised and semi-supervised learning (drill)

**Style B: blind-code drill.** Write each piece from memory, run the check, delete your code.
Reference: `../solutions/drills/03_unsupervised.py`.

**Traces to** `notes/ioai-task-patterns.md` P7 · T25-ANTIQUE (clusters → pseudo-labels → classifier),
T24-O-HYPER (PCA features). K-means and PCA are syllabus "Both" items, so sections 1–2 are **from scratch in NumPy**.

Total suggested time: **55 min**.

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs, make_moons, load_digits
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
rng = np.random.default_rng(7)

## 1. K-means from scratch ⏱ 12 min

`kmeans(X, k, n_iter=100, seed=0) -> (centers, labels)`, NumPy only: initialise (k-means++ or random points),
then alternate the assignment and update steps until the labels stop changing. Handle empty clusters.
The check needs ARI ≥ 0.95 on blobs and an inertia within 10 % of sklearn's.

In [ ]:
Xb, yb = make_blobs(n_samples=600, centers=5, cluster_std=1.0, random_state=3)
C, L = kmeans(Xb, 5)
C, L = np.asarray(C), np.asarray(L)
assert C.shape == (5, 2) and L.shape == (600,)
inertia = ((Xb - C[L]) ** 2).sum()
sk = KMeans(5, n_init=10, random_state=0).fit(Xb).inertia_
assert adjusted_rand_score(yb, L) >= 0.95 and inertia <= 1.10 * sk, (adjusted_rand_score(yb, L), inertia, sk)
print(f"section 1 ok (inertia {inertia:.1f} vs sklearn {sk:.1f})")

## 2. PCA from scratch ⏱ 10 min

- `pca_fit(X, k) -> (mean, components)`: `components` has shape `(k, d)`, orthonormal rows, ordered by variance.
  Use the SVD of the centred data.
- `pca_transform(X, mean, components) -> Z` of shape `(n, k)`.

In [ ]:
Xd = load_digits().data / 16.0
mean, comps = pca_fit(Xd, 8)
Zm = pca_transform(Xd, mean, comps)
ref = PCA(8).fit(Xd)
assert comps.shape == (8, 64) and Zm.shape == (len(Xd), 8)
assert np.allclose(comps @ comps.T, np.eye(8), atol=1e-6), "components must be orthonormal"
assert np.allclose(np.abs(comps @ ref.components_.T), np.eye(8), atol=1e-4), "wrong directions or order"
assert np.allclose(np.abs(Zm), np.abs(ref.transform(Xd)), atol=1e-4)
print("section 2 ok")

## 3. Best cluster-to-class matching ⏱ 8 min

`cluster_accuracy(y_true, clusters) -> float`: accuracy under the best **one-to-one** mapping from cluster ids
to class ids (Hungarian algorithm on the contingency table; `scipy.optimize.linear_sum_assignment`).

In [ ]:
yt = np.array([0, 0, 1, 1, 2, 2, 2])
assert np.isclose(cluster_accuracy(yt, [5, 5, 9, 9, 7, 7, 7]), 1.0)
assert np.isclose(cluster_accuracy(yt, [1, 1, 1, 0, 0, 2, 2]), 5 / 7)
assert np.isclose(cluster_accuracy(yt, [0, 0, 0, 0, 0, 0, 0]), 3 / 7)
print("section 3 ok")

## 4. Clustering two moons ⏱ 7 min

`moons_clusters(X) -> labels`: cluster the 2-D points into the two moons with any sklearn method
(not K-means). Needs ARI ≥ 0.95.

In [ ]:
Xm, ym = make_moons(800, noise=0.07, random_state=11)
ari = adjusted_rand_score(ym, moons_clusters(Xm))
assert ari >= 0.95, ari
print(f"section 4 ok (ARI {ari:.3f})")

## 5. Semi-supervised with 5 % labels ⏱ 10 min

`semi_supervised_predict(X, y_partial) -> y_full`: `y_partial` holds the class for known points and **`-1`** for
unknown ones. Return a label for every point. Needs ≥ 0.88 accuracy on the unknown digits.

In [ ]:
d = load_digits()
known = np.random.default_rng(0).random(len(d.target)) < 0.05
y_partial = np.where(known, d.target, -1)
pred = np.asarray(semi_supervised_predict(d.data, y_partial))
acc = (pred[~known] == d.target[~known]).mean()
assert pred.shape == d.target.shape and acc >= 0.88, acc
print(f"section 5 ok (accuracy on unknown {acc:.3f})")

## 6. A 2-D embedding that keeps the classes apart ⏱ 8 min

`embed_2d(X) -> E` of shape `(n, 2)`, using t-SNE or UMAP. A 5-NN classifier in your 2-D space needs ≥ 0.90
cross-validated accuracy on digits.

In [ ]:
E = np.asarray(embed_2d(d.data))
assert E.shape == (len(d.data), 2)
acc = cross_val_score(KNeighborsClassifier(5), E, d.target, cv=5).mean()
assert acc >= 0.90, acc
print(f"section 6 ok (5-NN accuracy in 2-D {acc:.3f})")